**Import dataset**

In [ ]:
import pandas as pd
import ast

df = pd.read_csv("data.csv")

df = df.dropna(subset=['clean_labels'])

# แปลง label string -> list
df["clean_labels"] = df["clean_labels"].apply(ast.literal_eval)

texts = df["Abstracts"].astype(str)
labels = df["clean_labels"]

**การจัดการ Label (MultiLabelBinarizer)** : เปลี่ยนชื่อหมวดหมู่ที่เดิมเป็นรูปแบบ String ให้กลายเป็นลักษณะของตัวเลข (One-hot encoding vector)

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
y = mlb.fit_transform(labels)

num_classes = len(mlb.classes_)

print("Classes:", mlb.classes_)

Classes: ['cs.AI' 'cs.CV' 'cs.LG' 'stat.ML']


**การทำ Tokenization (BERT Tokenizer)** : (Tokenize) และแปลงคำศัพท์เป็น ID ตามพจนานุกรมของ bert-base-uncased


*   AX_LEN = 128: กำหนดความยาวสูงสุดของประโยคที่โมเดลจะรับ
*   padding=True: ทำให้ทุกประโยคมีความยาวเท่ากัน (128 คำ)
* truncation=True: ตัดส่วนที่เกิน 128 คำออก เพื่อป้องกันไม่ให้ Memory เต็ม



In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

MAX_LEN = 256

encodings = tokenizer(
    texts.tolist(),
    padding=True,
    truncation=True,
    max_length=MAX_LEN,
    return_tensors="pt"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


**PaperDataset(Dataset)** : นี่คือการสร้างโครงสร้างข้อมูลแบบ Custom เพื่อให้ PyTorch ดึงข้อมูลไปใช้ได้อย่างเป็นระเบียบ


*   __init__: ทำหน้าที่รับข้อมูล encodings (ที่ได้จาก BERT) และ labels มาเก็บไว้ในตัวแปรของคลาส
*   __getitem__: หัวใจหลักของส่วนนี้ คือการบอกว่า "ถ้าต้องการข้อมูลลำดับที่ idx ให้ส่งอะไรกลับไป


*   __len__: บอกขนาดของชุดข้อมูลทั้งหมด เพื่อให้โมเดลรู้ว่า 1 รอบการเทรน (Epoch) ต้องดึงข้อมูลกี่ครั้ง




In [ ]:
import torch
from torch.utils.data import Dataset

class PaperDataset(Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels).float()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

In [ ]:
from sklearn.model_selection import train_test_split

train_idx, test_idx = train_test_split(
    range(len(texts)),
    test_size=0.2,
    random_state=42
)

train_enc = {k:v[train_idx] for k,v in encodings.items()}
test_enc = {k:v[test_idx] for k,v in encodings.items()}

train_labels = y[train_idx]
test_labels = y[test_idx]

**DataLoader**

In [ ]:
from torch.utils.data import DataLoader

train_dataset = PaperDataset(train_enc, train_labels)
test_dataset = PaperDataset(test_enc, test_labels)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

# **Train model**

In [ ]:
import torch.nn as nn
import torch

class TextCNN(nn.Module):

    def __init__(self, vocab_size, embed_dim, num_classes):

        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)

        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, 100, kernel_size=3),
            nn.Conv1d(embed_dim, 100, kernel_size=4),
            nn.Conv1d(embed_dim, 100, kernel_size=5)
        ])

        self.dropout = nn.Dropout(0.5)

        self.fc = nn.Linear(300, num_classes)

    def forward(self, input_ids):

        x = self.embedding(input_ids)

        x = x.permute(0,2,1)

        convs = [torch.relu(conv(x)) for conv in self.convs]

        pools = [torch.max(c, dim=2)[0] for c in convs]

        x = torch.cat(pools, dim=1)

        x = self.dropout(x)

        logits = self.fc(x)

        return logits

In [ ]:
vocab_size = tokenizer.vocab_size

model = TextCNN(
    vocab_size=vocab_size,
    embed_dim=128,
    num_classes=num_classes
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

TextCNN(
  (embedding): Embedding(30522, 128)
  (convs): ModuleList(
    (0): Conv1d(128, 100, kernel_size=(3,), stride=(1,))
    (1): Conv1d(128, 100, kernel_size=(4,), stride=(1,))
    (2): Conv1d(128, 100, kernel_size=(5,), stride=(1,))
  )
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=300, out_features=4, bias=True)
)

In [ ]:
import torch.optim as optim

criterion = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(model.parameters(), lr=2e-4)

In [ ]:
for epoch in range(100):

    model.train()
    total_loss = 0

    for batch in train_loader:

        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        logits = model(input_ids)

        loss = criterion(logits, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss:", total_loss/len(train_loader))

Epoch 1 Loss: 0.4737790336508846
Epoch 2 Loss: 0.37715111811920793
Epoch 3 Loss: 0.3526611646348742
Epoch 4 Loss: 0.3359186865844077
Epoch 5 Loss: 0.3282955066156639
Epoch 6 Loss: 0.31676656419647414
Epoch 7 Loss: 0.3089297082135235
Epoch 8 Loss: 0.302266980167216
Epoch 9 Loss: 0.29693446179825655
Epoch 10 Loss: 0.28898233946570206
Epoch 11 Loss: 0.2839766766421252
Epoch 12 Loss: 0.2756969251105903
Epoch 13 Loss: 0.2699332063046979
Epoch 14 Loss: 0.26377622942776885
Epoch 15 Loss: 0.2553440453353482
Epoch 16 Loss: 0.2480285871607807
Epoch 17 Loss: 0.23887662433017867
Epoch 18 Loss: 0.22973047911753536
Epoch 19 Loss: 0.2186210699939112
Epoch 20 Loss: 0.21078777205671223
Epoch 21 Loss: 0.2015537828023375
Epoch 22 Loss: 0.1890954327913867
Epoch 23 Loss: 0.1777732148329277
Epoch 24 Loss: 0.16681535544634704
Epoch 25 Loss: 0.15600045860915537
Epoch 26 Loss: 0.14515243938734904
Epoch 27 Loss: 0.13505292135758648
Epoch 28 Loss: 0.12678326266717183
Epoch 29 Loss: 0.11847899139731982
Epoch 30 L

# **Model Evaluation**

In [ ]:
model.eval()

all_preds = []

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch["input_ids"].to(device)

        logits = model(input_ids)

        probs = torch.sigmoid(logits)

        preds = (probs > 0.5).int()

        all_preds.append(preds.cpu())

In [ ]:
from sklearn.metrics import f1_score
import torch

preds = torch.cat(all_preds).numpy()

f1_micro = f1_score(test_labels, preds, average="micro")

print("Micro F1:", f1_micro)

Micro F1: 0.8486020226055919
